# 📈 ESG Regression Modeling
**Objective:** Predict Profit Margin from ESG and environmental metrics.

**Features:** ESG_Overall, CarbonEmissions, WaterUsage, EnergyConsumption

**Models:** Linear, Polynomial, Ridge, Lasso, ElasticNet, Random Forest, Gradient Boosting, XGBoost

---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib, os

print('Libraries loaded successfully!')

## 2. Load & Inspect Data

In [ ]:
try:
 df = pd.read_csv('esg_processed.csv')
 print('Loaded esg_processed.csv')
except FileNotFoundError:
 df = pd.read_csv('raw_data.csv')
 print('Loaded raw_data.csv')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n=== Target Variable Stats ===')
print(df['ProfitMargin'].describe())

## 3. Exploratory Analysis of Target & Features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(df['ProfitMargin'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('ProfitMargin Distribution')
axes[0].set_xlabel('Profit Margin')

# Boxplot by Industry
df.boxplot(column='ProfitMargin', by='Industry', ax=axes[1], rot=45)
axes[1].set_title('ProfitMargin by Industry')
axes[1].set_xlabel('')
plt.suptitle('')
plt.tight_layout()
plt.savefig('outputs/regression_target_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
FEATURES = ['ESG_Overall', 'CarbonEmissions', 'WaterUsage', 'EnergyConsumption']
TARGET = 'ProfitMargin'

corr_data = df[FEATURES + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_data, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
 linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/regression_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocessing & Train-Test Split

In [ ]:
df_model = df[FEATURES + [TARGET]].dropna()
X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
 X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape}')

## 5. Model Training & Comparison

In [ ]:
models = {
 'Linear Regression': LinearRegression(),
 'Polynomial (deg=2)': Pipeline([
 ('poly', PolynomialFeatures(degree=2, include_bias=False)),
 ('lr', LinearRegression())
 ]),
 'Ridge': Ridge(alpha=1.0),
 'Lasso': Lasso(alpha=0.01),
 'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
 'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
 'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
 'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
}

results = []
trained_models = {}

for name, model in models.items():
 model.fit(X_train_scaled, y_train)
 y_pred = model.predict(X_test_scaled)
 rmse = np.sqrt(mean_squared_error(y_test, y_pred))
 mae = mean_absolute_error(y_test, y_pred)
 r2 = r2_score(y_test, y_pred)
 results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})
 trained_models[name] = model
 print(f'{name:25s} | RMSE={rmse:.4f} | MAE={mae:.4f} | R²={r2:.4f}')

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
print('\n=== Ranked by R² ===')
results_df

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=['R² Score', 'RMSE', 'MAE'])

for i, metric in enumerate(['R2', 'RMSE', 'MAE'], 1):
 sorted_df = results_df.sort_values(metric, ascending=(metric != 'R2'))
 fig.add_trace(go.Bar(x=sorted_df['Model'], y=sorted_df[metric], name=metric,
 marker_color=['#2ecc71' if j == 0 else '#95a5a6' for j in range(len(sorted_df))]), row=1, col=i)

fig.update_layout(title='Regression Model Comparison', showlegend=False, height=450)
fig.show()

## 6. Hyperparameter Tuning (XGBoost)

In [ ]:
param_grid = {
 'n_estimators': [100, 200, 300],
 'max_depth': [3, 5, 7],
 'learning_rate': [0.01, 0.05, 0.1],
 'subsample': [0.8, 1.0]
}

xgb = XGBRegressor(random_state=42, verbosity=0)
grid_search = GridSearchCV(xgb, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_search.fit(X_train_scaled, y_train)

print(f'Best Params: {grid_search.best_params_}')
print(f'Best CV R²: {grid_search.best_score_:.4f}')
best_model = grid_search.best_estimator_

## 7. Final Model Evaluation

In [ ]:
y_pred_best = best_model.predict(X_test_scaled)

print('=== Tuned XGBoost Performance ===')
print(f'R² Score : {r2_score(y_test, y_pred_best):.4f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_best)):.4f}')
print(f'MAE : {mean_absolute_error(y_test, y_pred_best):.4f}')

In [ ]:
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_best, alpha=0.4, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_title('Actual vs Predicted')
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')

# Residuals vs Predicted
axes[1].scatter(y_pred_best, residuals, alpha=0.4, color='coral')
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_title('Residuals vs Predicted')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residuals')

# Residual Distribution
axes[2].hist(residuals, bins=50, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Residual Distribution')
axes[2].set_xlabel('Residual')

plt.tight_layout()
plt.savefig('outputs/regression_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
feat_imp = pd.DataFrame({
 'Feature': FEATURES,
 'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

fig = px.bar(feat_imp, x='Feature', y='Importance',
 color='Importance', color_continuous_scale='Blues',
 title='Feature Importance - XGBoost',
 text='Importance')
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.show()

## 8. Save Model & Results

In [ ]:
os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

joblib.dump(best_model, 'models/best_regression_model.pkl')
joblib.dump(scaler, 'models/regression_scaler.pkl')
results_df.to_csv('outputs/regression_results.csv', index=False)

# Save predictions
pred_df = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred_best, 'Residual': residuals.values})
pred_df.to_csv('outputs/regression_predictions.csv', index=False)

print('Saved:')
print(' - models/best_regression_model.pkl')
print(' - models/regression_scaler.pkl')
print(' - outputs/regression_results.csv')
print(' - outputs/regression_predictions.csv')